In [ ]:
# Import ESM classes from HF transformers
from transformers import EsmTokenizer, EsmForMaskedLM, AutoTokenizer
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
import torch.nn.functional as F
import random
import wandb
import argparse
import os
import time

In [ ]:
# Create the dataset to train on 
class UniRef50Dataset(Dataset):
    """
    Dataset class for UniRef50 protein sequences.

    This class loads sequences from a UniRef50 FASTA file and tokenizes them 
    using an ESM tokenizer for downstream model training. Sequences 
    can be circularly permuted with (`cp`). 

    Parameters
    ----------
    fasta_path : str
        Path to the UniRef50 FASTA file containing protein sequences.
    tokenizer_name : str
        Name of the pretrained ESM tokenizer to use (e.g. `"facebook/esm2_t33_650M_UR50D"`).
    max_length : int, default=1024
        Maximum sequence length (longer sequences will be truncated).
    cp : bool, default=True
        Whether to apply circular permutation to sequences.

    Attributes
    ----------
    sequences : list of str
        Raw protein sequences read from the FASTA file.
    tokenizer : EsmTokenizer
        Tokenizer used to convert protein sequences into tokens.
    max_length : int
        Maximum length cutoff for sequences.
    cp : bool
        circular permutation applied
    """
    def __init__(self, fasta_path, tokenizer_name, max_length, cp = True):
        self.sequences = self.read_fasta(fasta_path)
        self.tokenizer = EsmTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length
        self.cp = cp

    
    def read_fasta(self, fasta_path):
        ''' Get list of sequences from the fasta file 
        Input: fasta file path
        Returns: sequences from fasta file'''
        sequences = []
        seq = []
        with open(fasta_path) as f:
            for line in f:
                line = line.strip()
                if line.startswith(">"):
                    if seq:
                        sequences.append("".join(seq))
                        seq = []
                else:
                    seq.append(line)
            if seq:
                sequences.append("".join(seq))
                
            return sequences
            
    def __len__(self):
        ''' Returns number of sequences in the dataset '''
        return len(self.sequences)

    def __getitem__(self, idx):
        """
        Retrieve and tokenize a protein sequence.
    
        Parameters
        ----------
        idx : int
            Index of the sequence to retrieve.
    
        Returns
        -------
        dict
            A dictionary containing:
            
            - **input_ids** (torch.Tensor): Tokenized sequence IDs of shape `(L,)`,
              where `L <= max_length`.
            - **attention_mask** (torch.Tensor): Attention mask of shape `(L,)`
              indicating which tokens are padding (0) vs. real sequence (1).
        """
        seq = self.sequences[idx]

        # Create circular permutation of the sequence if cp == True
        if self.cp:
            i = random.randint(0, len(seq)-1)
            seq_cp = seq[i:] + seq[:i]

            #Tokenize the circular permutation
            tokenized = self.tokenizer(
                seq_cp,
                truncation=True,
                max_length=self.max_length,
                padding="max_length",
                return_tensors="pt"
            )
        
        else:
            #Tokenize the sequence
            tokenized = self.tokenizer(
                seq,
                truncation=True,
                max_length=self.max_length,
                padding="max_length",
                return_tensors="pt"
            )

        # Return a dictionary
        return {key: val.squeeze(0) for key, val in tokenized.items()}


In [ ]:
dataset = UniRef50Dataset(
    fasta_path='/home/ubuntu/uniref50.fasta',
    tokenizer_name='facebook/esm2_t30_150M_UR50D',
    max_length=128
)